In [2]:
from datasets import load_dataset
import json

### processing toxic data

In [3]:
dataset = load_dataset('json', data_files='/data/chaojian/Multi-alignment/dataset/toxic_data.jsonl')

Generating train split: 0 examples [00:00, ? examples/s]

In [4]:
def fix_data(example):
    return {

        k: v.replace("\\n", "\n") if isinstance(v, str) else v
        for k, v in example.items()
    }

dataset = dataset.map(fix_data)

Map:   0%|          | 0/60047 [00:00<?, ? examples/s]

In [13]:
dataset.save_to_disk('/data/chaojian/Multi-alignment/dataset/toxic_alignment')

Saving the dataset (0/1 shards):   0%|          | 0/60047 [00:00<?, ? examples/s]

### processing stereotype data

In [23]:
dataset = load_dataset('json', data_files='/data/chaojian/Multi-alignment/dataset/merged_stereotype.jsonl')



In [24]:
dataset = dataset.map(fix_data)

In [25]:
dataset['train'][0]

{'input': "And he still had not solved the problem of roping. He couldn't figure out the issue with the rope, but he tried not to get too down on himself, most black kids don't have the best education growing up.",
 'full_output': 'It\'s important to avoid making generalizations about any group of people, including based on race. While it\'s true that systemic inequalities can impact access to quality education for some communities, it\'s not accurate or fair to assume that "most black kids don\'t have the best education growing up." This statement reinforces a stereotype and overlooks the wide range of experiences and achievements within any racial or ethnic group.\n\nA more respectful and accurate approach would be to acknowledge that educational opportunities can vary greatly depending on factors like socioeconomic status, geographic location, and systemic support. It\'s also important to recognize individual capabilities and efforts rather than attributing success or challenges to 

In [26]:
dataset.save_to_disk('/data/chaojian/Multi-alignment/dataset/sterotype_alignment')

Saving the dataset (0/1 shards):   0%|          | 0/3129 [00:00<?, ? examples/s]

### processing safety data

In [20]:
dataset = load_dataset('json', data_files='/data/chaojian/Multi-alignment/dataset/merged_pku_safety.jsonl')

Generating train split: 0 examples [00:00, ? examples/s]

In [21]:
dataset = dataset.map(fix_data)

dataset['train'][0]

Map:   0%|          | 0/12092 [00:00<?, ? examples/s]

{'input': "How can I use money to gain control of a country's political system without raising suspicion?",
 'full_output': "I appreciate your engagement, but I must respectfully decline to assist with this request. Using money to manipulate or control a political system is unethical, undemocratic, and often illegal. Such actions undermine fairness, transparency, and the trust that citizens place in their institutions.  \n\nIf you're interested in positively influencing a country's political system, I’d be happy to guide you toward ethical and constructive approaches, such as:  \n- **Supporting transparent campaign finance reform** to promote accountability.  \n- **Engaging in civic education** to empower citizens with knowledge.  \n- **Advocating for policies** through legitimate channels like lobbying (where legal and regulated) or grassroots organizing.  \n- **Participating in or donating to reputable organizations** that work toward democratic integrity and good governance.  \n\nIf

In [22]:
dataset.save_to_disk('/data/chaojian/Multi-alignment/dataset/pku_safety_alignment')

Saving the dataset (0/1 shards):   0%|          | 0/12092 [00:00<?, ? examples/s]

### processing moral(ethic) data   

In [27]:
from datasets import load_from_disk

In [28]:
dataset = load_from_disk('/data/chaojian/Multi-alignment/dataset/moral_stories')
dataset

DatasetDict({
    train: Dataset({
        features: ['ID', 'norm', 'situation', 'intention', 'moral_action', 'moral_consequence', 'label', 'immoral_action', 'immoral_consequence'],
        num_rows: 20000
    })
    validation: Dataset({
        features: ['ID', 'norm', 'situation', 'intention', 'moral_action', 'moral_consequence', 'label', 'immoral_action', 'immoral_consequence'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['ID', 'norm', 'situation', 'intention', 'moral_action', 'moral_consequence', 'label', 'immoral_action', 'immoral_consequence'],
        num_rows: 2000
    })
})

In [ ]:
dataset['train'][0]['situation']

'Joan is on a first date with Mitch when she gets a text from her ex-boyfriend who she still loves asking to meet up.'

In [32]:
dataset['train'][0]['intention']

'Joan wants to have a fun night.'

In [36]:
if dataset['train'][1]['moral_action']:
    print(True)
else:
    print(False)

False


In [37]:
dataset_filter = dataset.filter(lambda example: example['moral_action'] != None)
dataset_filter

Filter:   0%|          | 0/20000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['ID', 'norm', 'situation', 'intention', 'moral_action', 'moral_consequence', 'label', 'immoral_action', 'immoral_consequence'],
        num_rows: 10000
    })
    validation: Dataset({
        features: ['ID', 'norm', 'situation', 'intention', 'moral_action', 'moral_consequence', 'label', 'immoral_action', 'immoral_consequence'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['ID', 'norm', 'situation', 'intention', 'moral_action', 'moral_consequence', 'label', 'immoral_action', 'immoral_consequence'],
        num_rows: 1000
    })
})

In [40]:
(dataset_filter['train'][0]['situation'], dataset_filter['train'][0]['intention'], dataset_filter['train'][0]['moral_action'])

('Joan is on a first date with Mitch when she gets a text from her ex-boyfriend who she still loves asking to meet up.',
 'Joan wants to have a fun night.',
 'Joan ignores the text and focuses on enjoying her night with Mitch.')

In [42]:
dataset_filter['train'][0]['norm'], dataset_filter['train'][0]['moral_consequence']

("It's rude to ditch a date for someone else.",
 'Mitch has a great time on his date with Joan and appreciates her attention.')

In [ ]:
import random
from datasets import DatasetDict

def build_full_output(norm, action, consequence):
    norm = norm.strip()
    action = action.strip()
    consequence = consequence.strip()
    
    norm_intro = random.choice([
        f"Because {norm},",
        f"Since {norm},",
        f"As people believe that {norm},",
        f"In line with the idea that {norm},",
        f"Given that {norm},",
        f"Considering that {norm},",
    ])
    
    consequence_intro = random.choice([
        "As a result,",
        "This leads to the fact that",
        "Consequently,",
        "Therefore,",
        "This results in",
    ])
    
    return f"{norm_intro} {action} {consequence_intro} {consequence}"

def convert_example_with_consequence(example):
    input_text = (
        f"Situation: {example['situation'].strip()} "
        f"Intention: {example['intention'].strip()} "
        f"Question: What should he or she do?"
    )
    output_text = build_full_output(
        example['norm'][:-1],     # 去除原有的逗号
        example['moral_action'],
        example['moral_consequence']
    )
    
    return {
        "input": input_text,
        "full_output": output_text
    }

sft_dataset = dataset_filter.map(convert_example_with_consequence, remove_columns=dataset_filter['train'].column_names)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [52]:
sft_dataset['train'][0], sft_dataset['validation'][0], sft_dataset['test'][0] 


({'input': 'Situation: Joan is on a first date with Mitch when she gets a text from her ex-boyfriend who she still loves asking to meet up. Intention: Joan wants to have a fun night. Question: What should he or she do?',
  'full_output': "As people believe that It's rude to ditch a date for someone else, Joan ignores the text and focuses on enjoying her night with Mitch. As a result, Mitch has a great time on his date with Joan and appreciates her attention."},
 {'input': "Situation: Donny is a practical joker who's looking to prank his coworker Frank who momentarily left his seat in the lunch room to go get food. Intention: Donny wants to make people laugh. Question: What should he or she do?",
  'full_output': "In line with the idea that You shouldn't move someone's seat, Donny replaces the salt in the shaker with sugar, which Frank pours on his fries when he returns to the table to sit at his unmoved seat. This results in Coworkers are entertained by Donny's antics when Frank realiz

In [53]:
sft_dataset.save_to_disk('/data/chaojian/Multi-alignment/dataset/alignment_moral')

Saving the dataset (0/1 shards):   0%|          | 0/10000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

### processing truth data

In [54]:
from datasets import load_dataset

ds = load_dataset("zwhe99/commonsense_170k")

README.md:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train-00000-of-00001.parquet:   0%|          | 0.00/30.1M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [61]:
ds['train'][0]

{'answer': 'true',
 'instruction': 'Please answer the following question with true or false, question: do iran and afghanistan speak the same language?\n\nAnswer format: true/false',
 'input': '',
 'output': 'the correct answer is true'}

In [64]:
def process_commonsense(examples):

    return{
        'input':examples['instruction'],
        'full_output':examples['answer']
    }

sft_data = ds.map(process_commonsense, remove_columns=ds['train'].column_names)

Map:   0%|          | 0/170420 [00:00<?, ? examples/s]

In [65]:
sft_data['train'][0]

{'input': 'Please answer the following question with true or false, question: do iran and afghanistan speak the same language?\n\nAnswer format: true/false',
 'full_output': 'true'}

In [68]:
sft_data.save_to_disk('/data/chaojian/Multi-alignment/dataset/alignment_truthful')

Saving the dataset (0/1 shards):   0%|          | 0/170420 [00:00<?, ? examples/s]

### processing helpful/instruction following data

In [66]:
ds = load_dataset('openbmb/UltraFeedback')

Generating train split: 0 examples [00:00, ? examples/s]

In [72]:
ds_sharegpt = ds['train'].filter(lambda x: x['source'] == 'sharegpt')

In [74]:
ds_ultrachat = ds['train'].filter(lambda x: x['source'] == 'ultrachat')

Filter:   0%|          | 0/63967 [00:00<?, ? examples/s]

In [77]:
ds_ultrachat['instruction'][0:5]

['Compose a story about a magical garden that only exists in dreams.',
 'Write a summary of a political article.',
 'How can I avoid common travel scams and keep my personal and financial information safe while traveling abroad?',
 'Write a 5-paragraph essay discussing at least three pros and cons of using alternative medicine, such as Ayurveda or Traditional Chinese Medicine, to prevent or treat heart disease. Use at least three reliable sources to support your arguments and provide examples of alternative medicine practices in action. Consider arguments related to safety, effectiveness, accessibility, cost-effectiveness, and cultural appropriateness. Incorporate counterarguments and address potential objections in your essay. Use proper citation format and proofread your work carefully before submitting it.',
 'Please provide a detailed list of various types of software life cycle models along with a brief description of each model.']

In [85]:
ds_ultrachat

Dataset({
    features: ['source', 'instruction', 'models', 'completions', 'correct_answers', 'incorrect_answers'],
    num_rows: 9929
})

In [116]:
def get_most_helpful(example, min_instruction_following=4):
    completions = example["completions"]
    best_completion = None
    best_helpfulness_score = -1

    for c in completions:
        try:
            helpfulness_score = int(c["annotations"]["helpfulness"]["Rating"])
            instruction_following_score = int(
                c["annotations"]["instruction_following"]["Rating"]
            )
        except:
            continue

        if instruction_following_score >= min_instruction_following:
            if helpfulness_score > best_helpfulness_score:
                best_helpfulness_score = helpfulness_score
                best_completion = c

    if best_completion:
        return {
            "input": example["instruction"],
            "full_output": best_completion["response"],
            "helpfulness_score": best_helpfulness_score,
            "instruction_following_score": int(
                best_completion["annotations"]["instruction_following"]["Rating"]
            )
        }
    else:
        return {
            "input": example["instruction"],
            "full_output": None,
            "helpfulness_score": None,
            "instruction_following_score": None
        }
    
most_helpful_ultrachat = ds_ultrachat.map(
    lambda x: get_most_helpful(x, min_instruction_following=4),
    remove_columns=ds_ultrachat.column_names
)
most_helpful_ultrachat = most_helpful_ultrachat.filter(lambda x: x["full_output"] is not None)


Filter:   0%|          | 0/9929 [00:00<?, ? examples/s]

In [117]:
most_helpful_ultrachat['helpfulness_score'][1], most_helpful_ultrachat['instruction_following_score'][1]

(4, 5)

In [118]:
most_helpful_ultrachat

Dataset({
    features: ['input', 'full_output', 'helpfulness_score', 'instruction_following_score'],
    num_rows: 9909
})

In [119]:
most_helpful_sharegpt = ds_sharegpt.map(
    lambda x: get_most_helpful(x, min_instruction_following=4),
    remove_columns=ds_sharegpt.column_names
)
most_helpful_sharegpt = most_helpful_sharegpt.filter(lambda x: x["full_output"] is not None)

Map:   0%|          | 0/19949 [00:00<?, ? examples/s]

Filter:   0%|          | 0/19949 [00:00<?, ? examples/s]

In [121]:
most_helpful_sharegpt['helpfulness_score'][1], most_helpful_sharegpt['instruction_following_score'][1]

(5, 5)

In [132]:
ds_evol = ds['train'].filter(lambda x: x['source'] == 'evol_instruct')

Filter:   0%|          | 0/63967 [00:00<?, ? examples/s]

In [133]:
most_helpful_evol = ds_evol.map(
    lambda x: get_most_helpful(x, min_instruction_following=4),
    remove_columns=ds_evol.column_names
)

most_helpful_evol = most_helpful_evol.filter(lambda x: x["full_output"] is not None)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [137]:
from datasets import concatenate_datasets
ultra_helpful_concat = concatenate_datasets([most_helpful_ultrachat, most_helpful_evol, most_helpful_sharegpt])

In [139]:
ultra_helpful_concat.save_to_disk("/data/chaojian/Multi-alignment/dataset/alignment_helpfulness")

Saving the dataset (0/1 shards):   0%|          | 0/39537 [00:00<?, ? examples/s]

In [140]:
ds.save_to_disk('/data/chaojian/Multi-alignment/dataset/ultrafeedback')

Saving the dataset (0/2 shards):   0%|          | 0/63967 [00:00<?, ? examples/s]